# Phase 4: Privacy Mask Prescription Images

This notebook manually masks private information in prescription images before using them in thesis/report/PPT/demo outputs.

Input folder: `data/raw`

Output folder: `data/masked`

Mask patient details, doctor/clinic identifiers, signatures, seals, phone numbers, addresses, QR/barcodes, IDs, and dates if not needed. Keep medicine handwriting visible.

## Step 1: Mount Drive and Enter Repo

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = 'https://github.com/nbl-ahmd/project.git'
DRIVE_BASE = Path('/content/drive/MyDrive/phase4_project')
REPO_DIR = DRIVE_BASE / 'repo'
IN_COLAB = 'google.colab' in sys.modules or 'COLAB_GPU' in os.environ

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_BASE.mkdir(parents=True, exist_ok=True)
    if not (REPO_DIR / 'pipeline').exists():
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=False)
else:
    REPO_DIR = Path.cwd()

os.chdir(REPO_DIR)
print('Repository:', Path.cwd())
print('Has data/raw:', (Path.cwd() / 'data' / 'raw').exists())

## Step 2: Install/Import Tools

In [ ]:
!python3 -m pip install -q opencv-python pillow ipywidgets pandas

from pathlib import Path
import json
import cv2
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import ipywidgets as widgets

## Step 3: Configure Input and Output Folders

In [ ]:
RAW_DIR = Path('data/raw')
MASKED_DIR = Path('data/masked')
MASKED_DIR.mkdir(parents=True, exist_ok=True)

RECTANGLES_JSON = MASKED_DIR / 'mask_rectangles.json'
MANIFEST_CSV = MASKED_DIR / 'mask_manifest.csv'

VALID_EXTS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp', '.tif', '.tiff'}
image_paths = sorted([p for p in RAW_DIR.iterdir() if p.is_file() and p.suffix.lower() in VALID_EXTS])

print('Raw folder:', RAW_DIR.resolve())
print('Masked folder:', MASKED_DIR.resolve())
print('Images found:', len(image_paths))
for p in image_paths[:10]:
    print('-', p.name)

## Step 4: Interactive Rectangle Masker

Use sliders to select a rectangle. Click **Add Rectangle**. Add as many rectangles as needed. Click **Save Masked Image** to write the masked copy.

Use black rectangles, not blur.

In [ ]:
if not image_paths:
    raise FileNotFoundError(f'No images found in {RAW_DIR}')

saved_rectangles = {}
if RECTANGLES_JSON.exists():
    saved_rectangles = json.loads(RECTANGLES_JSON.read_text(encoding='utf-8'))

state = {
    'idx': 0,
    'rects': [],
}

image_dropdown = widgets.Dropdown(
    options=[(f'{i}: {p.name}', i) for i, p in enumerate(image_paths)],
    value=0,
    description='Image:',
    layout=widgets.Layout(width='90%'),
)
x1_slider = widgets.IntSlider(description='x1', min=0, max=100, value=0, continuous_update=False)
y1_slider = widgets.IntSlider(description='y1', min=0, max=100, value=0, continuous_update=False)
x2_slider = widgets.IntSlider(description='x2', min=0, max=100, value=100, continuous_update=False)
y2_slider = widgets.IntSlider(description='y2', min=0, max=100, value=100, continuous_update=False)
add_button = widgets.Button(description='Add Rectangle', button_style='warning')
undo_button = widgets.Button(description='Undo Last', button_style='')
clear_button = widgets.Button(description='Clear Rectangles', button_style='')
save_button = widgets.Button(description='Save Masked Image', button_style='success')
next_button = widgets.Button(description='Next Image', button_style='info')
status = widgets.HTML()
out = widgets.Output()

def load_image(path):
    img = cv2.imread(str(path))
    if img is None:
        raise ValueError(f'Could not read {path}')
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

def current_path():
    return image_paths[state['idx']]

def set_slider_limits(img):
    h, w = img.shape[:2]
    for slider, max_value in [(x1_slider, w - 1), (x2_slider, w - 1)]:
        slider.max = max_value
    for slider, max_value in [(y1_slider, h - 1), (y2_slider, h - 1)]:
        slider.max = max_value
    x1_slider.value = 0
    y1_slider.value = 0
    x2_slider.value = min(w - 1, max(100, w // 3))
    y2_slider.value = min(h - 1, max(100, h // 8))

def normalize_rect(rect):
    x1, y1, x2, y2 = [int(v) for v in rect]
    return [min(x1, x2), min(y1, y2), max(x1, x2), max(y1, y2)]

def draw_preview(img, rects, active_rect=None):
    preview = img.copy()
    for rect in rects:
        x1, y1, x2, y2 = normalize_rect(rect)
        preview[y1:y2, x1:x2] = 0
        cv2.rectangle(preview, (x1, y1), (x2, y2), (255, 0, 0), 3)
    if active_rect is not None:
        x1, y1, x2, y2 = normalize_rect(active_rect)
        cv2.rectangle(preview, (x1, y1), (x2, y2), (0, 255, 255), 3)
    return preview

def refresh():
    path = current_path()
    img = load_image(path)
    active = [x1_slider.value, y1_slider.value, x2_slider.value, y2_slider.value]
    preview = draw_preview(img, state['rects'], active)
    with out:
        clear_output(wait=True)
        plt.figure(figsize=(10, 12))
        plt.imshow(preview)
        plt.title(f'{state["idx"] + 1}/{len(image_paths)}: {path.name}')
        plt.axis('off')
        plt.show()
    status.value = f'<b>{path.name}</b> | rectangles: {len(state["rects"])} | output: {MASKED_DIR / path.name}'

def load_current_rects():
    path = current_path()
    state['rects'] = saved_rectangles.get(path.name, []).copy()
    img = load_image(path)
    set_slider_limits(img)
    refresh()

def on_image_change(change):
    state['idx'] = int(change['new'])
    load_current_rects()

def on_slider_change(change):
    refresh()

def add_rect(_):
    state['rects'].append(normalize_rect([x1_slider.value, y1_slider.value, x2_slider.value, y2_slider.value]))
    refresh()

def undo_rect(_):
    if state['rects']:
        state['rects'].pop()
    refresh()

def clear_rects(_):
    state['rects'] = []
    refresh()

def save_masked(_):
    path = current_path()
    img = load_image(path)
    masked = img.copy()
    for rect in state['rects']:
        x1, y1, x2, y2 = normalize_rect(rect)
        masked[y1:y2, x1:x2] = 0
    out_path = MASKED_DIR / path.name
    cv2.imwrite(str(out_path), cv2.cvtColor(masked, cv2.COLOR_RGB2BGR))
    saved_rectangles[path.name] = state['rects'].copy()
    RECTANGLES_JSON.write_text(json.dumps(saved_rectangles, indent=2), encoding='utf-8')
    manifest_rows = []
    for name, rects in saved_rectangles.items():
        manifest_rows.append({'source_image': str(RAW_DIR / name), 'masked_image': str(MASKED_DIR / name), 'rectangle_count': len(rects), 'rectangles': json.dumps(rects)})
    pd.DataFrame(manifest_rows).to_csv(MANIFEST_CSV, index=False)
    status.value = f'<b>Saved:</b> {out_path} | rectangles: {len(state["rects"])}'
    refresh()

def next_image(_):
    if state['idx'] < len(image_paths) - 1:
        state['idx'] += 1
        image_dropdown.value = state['idx']
        load_current_rects()

image_dropdown.observe(on_image_change, names='value')
for slider in [x1_slider, y1_slider, x2_slider, y2_slider]:
    slider.observe(on_slider_change, names='value')
add_button.on_click(add_rect)
undo_button.on_click(undo_rect)
clear_button.on_click(clear_rects)
save_button.on_click(save_masked)
next_button.on_click(next_image)

display(image_dropdown)
display(widgets.HBox([x1_slider, x2_slider]))
display(widgets.HBox([y1_slider, y2_slider]))
display(widgets.HBox([add_button, undo_button, clear_button, save_button, next_button]))
display(status)
display(out)

load_current_rects()

## Step 5: Review Saved Masked Images

In [ ]:
from IPython.display import display, Image as IPImage

masked_images = sorted([p for p in MASKED_DIR.iterdir() if p.is_file() and p.suffix.lower() in VALID_EXTS])
print('Masked images saved:', len(masked_images))
print('Manifest:', MANIFEST_CSV, 'exists=', MANIFEST_CSV.exists())

for p in masked_images[:10]:
    print(p)
    display(IPImage(filename=str(p), width=700))

## Step 6: Use Masked Images in Pipeline

After masking, use `data/masked` instead of `data/raw` in inference/segmentation notebooks.

Example:

```python
INPUT_PATH = Path('data/masked')
```

Do not upload unmasked full prescriptions to GitHub, papers, slides, or public datasets.